<a href="https://colab.research.google.com/github/ThanuraTG/Carecloud_Medicine_System/blob/main/DS_Mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ***Clean and Preprocess data***

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [21]:
# Load the data
df = pd.read_csv('/content/weatherHistory.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/weatherHistory.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# 1. Remove unnecessary first column (empty index column and salad column)
df = df.drop(columns=['Loud Cover', 'Formatted Date', 'Daily Summary'])

In [ ]:
# 2. Standardize Summary and Precip Type names
df['Summary'] = df['Summary'].str.strip()
df['Precip Type'] = df['Precip Type'].str.strip()

In [ ]:
# 3. Clean Precip Type names
df['Precip Type'] = df['Precip Type'].str.strip()


In [22]:
# 4. Handle missing values (NA)
# 4.1 For numeric columns, replace NA with NaN
numeric_cols = ['Temperature (C)', 'Apparent Temperature (C)', 'Humidity',	'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)']
df[numeric_cols] = df[numeric_cols].replace('NA', pd.NA)

NameError: name 'df' is not defined

In [ ]:
# 4.2 Create a cleaned version of the dataset
cleaned_df = df.copy()

In [ ]:
# 4.3 Check for missing values
print("Missing values per column: ")
print(cleaned_df.isnull().sum())

In [ ]:
# 4.4 clean missing values
cleaned_df.fillna(0,inplace=True)

In [23]:
# 4.5 Check for missing values
print("Missing values per column: ")
print(cleaned_df.isnull().sum())

Missing values per column: 


NameError: name 'cleaned_df' is not defined

In [ ]:
# 5. Convert columns to numeric type
cleaned_df[numeric_cols] = cleaned_df[numeric_cols].apply(pd.to_numeric)

In [ ]:
# 6
# 6.1 Check for duplicates
print(f"Duplicate rows: {cleaned_df.duplicated().sum()}")

In [ ]:
# 6.2 clean duplicates values
cleaned_df.drop_duplicates(inplace=True)

In [ ]:
# 6.3 Check for duplicates
print(f"Duplicate rows: {cleaned_df.duplicated().sum()}")


In [ ]:
# 6.4 cleaned shape
cleaned_df.shape

## ***Exploratory Data Analysis***

In [ ]:
# 1. Basic statistics
print("\nBasic statistics:")
print(cleaned_df[numeric_cols].describe())

In [ ]:
# 2. Temperature Distribution Analysis
plt.figure(figsize=(10, 6))
sns.histplot(data=cleaned_df,x='Temperature (C)', bins=30, kde=True)
plt.title("Distribution of Temperature (C)")
plt.xlabel("Temperature (C)")
plt.ylabel("Frequency")
plt.show()

In [24]:
# Boxplot of Temperature
plt.figure(figsize=(10, 6)) # Adjust dimensions for horizontal plot
sns.boxplot(data=cleaned_df['Temperature (C)'], orient='h') # 'h' for horizontal
plt.title("Boxplot of Temperature (C)")
plt.show()

NameError: name 'cleaned_df' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
# 3.1 Temperature by Chain:
avg_calories = cleaned_df.groupby('Summary')['Temperature (C)'].mean().sort_values()
avg_calories

In [ ]:
# 3.2 Calories by plot:
avg_calories.plot(kind='bar', figsize=(10, 5))
plt.title("Average Temperature per Summary")
plt.ylabel("Temperature")
plt.show()

In [ ]:
# 4. Principal Component Analysis (PCA):
# 4.1 Select relevant nutritional variables (excluding categorical variables. calories and )
nutrition_cols = ['Temperature (C)','Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)','Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)']

In [ ]:
# 4.2 Create a subset with only nutritional data
nutrition_df = cleaned_df[nutrition_cols]

In [25]:
# 4.3 Standardize the data (important for PCA)
scaler = StandardScaler()
nutrition_scaled = scaler.fit_transform(nutrition_df)

NameError: name 'nutrition_df' is not defined

In [ ]:
# 4.4 Perform PCA
pca = PCA()
principal_components = pca.fit_transform(nutrition_scaled)

In [ ]:
# 4.5 Eigenvalues (explained variance)
eigenvalues = pca.explained_variance_
print("Eigenvalues (Explained Variance):")
print(eigenvalues)
print('\n')

In [ ]:
# 4.6 Calculate total variation
total_variation = sum(eigenvalues)
print(f"Total Variation: {total_variation:.4f}")

In [26]:
# 4.7 Scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(eigenvalues)+1), eigenvalues, 'bo-', linewidth=2)
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue (Explained Variance)')
plt.axhline(y=1, color='r', linestyle='--') # Kaiser criterion line
plt.grid()
plt.show()


NameError: name 'eigenvalues' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
# 4.8 Eigenvectors (loadings)
eigenvectors = pca.components_
print("\nEigenvectors (Principal Components Loadings):")
for i, component in enumerate(eigenvectors):
    print(f"\nPrincipal Component {i+1}:")
    print(pd.Series(component, index=nutrition_cols))

In [ ]:
# 5. Restaurant-wise Data Balance
print(cleaned_df["Summary"].value_counts())

In [ ]:
# 6.1 Count plot for restaurant distribution
plt.figure(figsize=(12, 6))
sns.countplot(data=cleaned_df, x='Summary', order=df['Summary'].value_counts().iloc[:10].index)
plt.xticks(rotation=45)
plt.title("Top 10 Weather Summaries")
plt.xlabel("Summary")
plt.ylabel("Count")
plt.show()

In [ ]:
# 6.2 Comparison of Nutrients
data = cleaned_df.copy()
data['Humidity'] = data['Humidity'] * 10
top_summaries = data['Summary'].value_counts().nlargest(5).index
filtered_data = data[data['Summary'].isin(top_summaries)]
weather_stats = filtered_data.groupby('Summary')[['Temperature (C)', 'Wind Speed (km/h)', 'Visibility (km)']].mean()
print(weather_stats)

plt.figure(figsize=(12, 6))
weather_stats.plot(kind='bar', stacked=False)

plt.title('Weather Condition-wise Comparison of Factors')
plt.ylabel('Average Value')
plt.xticks(rotation=45)
plt.legend(title='Factors')
plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Correlation heatmap
numeric_cols = ['Temperature (C)', 'Humidity', 'Wind Speed (km/h)' , 'Visibility (km)', 'Pressure (millibars)']

plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0 )
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# 6.4 Set up the figure
plt.figure(figsize=(18, 5))

# Plot 1: Calories vs Calories from Fat
plt.subplot(1, 3, 3)
sns.scatterplot(x='Humidity', y='Temperature (C)', data=cleaned_df, hue='Summary', palette='viridis', alpha=0.5)
plt.title('Temperature vs Humidity')
plt.xlabel('Humidity')
plt.ylabel('Temperature (C)')
plt.legend().remove()

# Plot 2: Calories vs Cholesterol
plt.subplot(1, 3, 2)
sns.scatterplot(x='Wind Speed (km/h)', y='Temperature (C)', data=cleaned_df, hue='Summary', palette='viridis',  alpha=0.5)
plt.title('Temperature vs Wind Speed')
plt.xlabel('Wind Speed (km/h)')
plt.ylabel('Temperature (C)')
plt.legend().remove()

# Plot 3: Calories vs Protein
plt.subplot(1, 3, 1)
sns.scatterplot(x='Pressure (millibars)', y='Temperature (C)', data=cleaned_df, hue='Summary', palette='viridis', alpha=0.5)
plt.title('Temperature vs Pressure')
plt.xlabel('Pressure (millibars)')
plt.ylabel('Temperature (C)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## ***ML/AI methods***

In [ ]:
# Select features and target
X = data[['Humidity','Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)','Pressure (millibars)', 'Apparent Temperature (C)']]
y = data['Temperature (C)']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=88)

### Linear Regression

In [ ]:
# 1. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

In [ ]:
print("\nLinear Regression Performance:")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("MSE:", mean_squared_error(y_test, lr_pred))
print("R2 Score:", r2_score(y_test, lr_pred))

### Random Forest Regressor

In [ ]:
# 2. Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=88)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [ ]:
print("\nRandom Forest Performance:")
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("R2 Score:", r2_score(y_test, rf_pred))